In [2]:
import pandas as pd
import numpy as np
import csv
pd.set_option('display.max_colwidth', None)

### Загружаем файлы и сводим в один

In [2]:
df = pd.read_excel(r"D:\Ricci\archive\realty_sold_07052026_M.xlsx", sheet_name="Данные")

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 862098 entries, 0 to 862097
Data columns (total 62 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   ID ЖК                            857787 non-null  float64       
 1   ЖК рус                           857787 non-null  object        
 2   ЖК англ                          364027 non-null  object        
 3   Район Город                      857787 non-null  object        
 4   Округ Направление                857787 non-null  object        
 5   Регион                           857787 non-null  object        
 6   АТД                              857787 non-null  object        
 7   Застройщик ЖК                    857787 non-null  object        
 8   Описание помещения               857787 non-null  object        
 9   Площадь                          857751 non-null  float64       
 10  Комнатность                      139929 non-

In [3]:
df2 = pd.read_excel(r"D:\Ricci\archive\realty_sold_07052026_MO.xlsx", sheet_name="Данные")

In [5]:
df3 = pd.read_excel(r"D:\Ricci\archive\realty_sold_07052026_NM.xlsx", sheet_name="Данные")

In [50]:
df_общий = pd.concat([df, df2, df3], ignore_index=True)

In [118]:
df_общий.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1234375 entries, 0 to 1234374
Data columns (total 18 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   ID ЖК              1234375 non-null  object        
 1   ЖК рус             1234375 non-null  object        
 2   Район Город        1234375 non-null  object        
 3   Округ Направление  1234375 non-null  object        
 4   Регион             1234375 non-null  object        
 5   АТД                1234375 non-null  object        
 6   Застройщик ЖК      1234375 non-null  object        
 7   Площадь            1234375 non-null  float64       
 8   Тип Комнатности    1219776 non-null  object        
 9   Тип помещения      1234375 non-null  object        
 10  Корпус             1223919 non-null  object        
 11  Дата регистрации   1234375 non-null  datetime64[ns]
 12  Залогодержатель    785430 non-null   object        
 13  Тип обременения    786238 n

In [117]:
df_общий['ID дом.рф'] = df_общий['ID дом.рф'].apply(lambda x: str(int(x)) if pd.notna(x) else '')
df_общий['ID ЖК'] = df_общий['ID ЖК'].apply(lambda x: str(int(x)) if pd.notna(x) else '')


In [ ]:
df.isna().sum()

In [57]:
# Сначала убедимся, что все столбцы в числовом формате
# (пустые строки станут NaN)
for col in ['Оценка цены', 'Цена со скидкой', 'Оценка по ЕИСЖС']:
    df_общий[col] = pd.to_numeric(df_общий[col], errors='coerce')

# Заполняем пропуски: сначала из Цена со скидкой, затем из Оценка по ЕИСЖС
df_общий['Оценка цены'] = df_общий['Оценка цены'].fillna(df_общий['Цена со скидкой'])
df_общий['Оценка цены'] = df_общий['Оценка цены'].fillna(df_общий['Оценка по ЕИСЖС'])

In [45]:
df_общий['Уступка'].unique()

array([0.])

In [53]:
# на всякий случай убираем пробелы
df_общий.columns = df_общий.columns.str.strip()
# нужный порядок столбцов
columns_order = [
    "ID ЖК",
    "ЖК рус",
    "Район Город",
    "Округ Направление",
    "Регион",
    "АТД",
    "Застройщик ЖК",
    "Площадь",
    "Тип Комнатности",
    "Тип помещения",
    "Корпус",
    "Покупатель ЮЛ",
    "Дата регистрации",
    "Залогодержатель",
    "Тип обременения",
    "Оценка цены",
    "Цена со скидкой",
    "Оценка по ЕИСЖС",
    "Уступка",
    "Купил лотов в ЖК",
    "Ипотека",
    "ID дом.рф"
]

# оставляем и упорядочиваем столбцы
df_общий = df_общий.reindex(columns=columns_order)

In [61]:
df_общий = df_общий[df_общий['Тип помещения'].isin(['квартира', 'апартамент'])]

In [62]:
df_общий = df_общий[df_общий['Дата регистрации'] > '2016-01-01']

In [35]:
yearly_counts = df_общий['Дата регистрации'].dt.year.value_counts().sort_index()
print(yearly_counts)

Дата регистрации
2016     78360
2017    108509
2018    149797
2019    175058
2020    175037
2021    182414
2022    141858
2023    175991
2024    144283
2025    147628
2026     31635
Name: count, dtype: int64


In [63]:
df_общий = df_общий[(df_общий['Покупатель ЮЛ'].isna()) | (df_общий['Покупатель ЮЛ'] == '')]

In [64]:
df_общий = df_общий[df_общий['Купил лотов в ЖК'] <= 5]

In [65]:
df_общий = df_общий[df_общий['Уступка'] == 0]

In [ ]:
df_общий.to_csv(r"D:\Ricci\Pipin_sales-05-2026.csv", index=False, encoding='utf-8-sig')

In [47]:
empty_count = df_общий['Оценка цены'].isna().sum() + (df_общий['Оценка цены'] == '').sum() + (df_общий['Оценка цены'].isnull()).sum()
print(empty_count)

24748


In [104]:
pivot_empty = df_общий[df_общий['Оценка цены'].isna()].pivot_table(
    index='Год',
    columns='Месяц',
    aggfunc='size',
    fill_value=0
)
print(pivot_empty)


Empty DataFrame
Columns: []
Index: []


In [94]:
# Сначала преобразуем столбец в числовой тип (пустые строки станут NaN)
df_общий['Площадь'] = pd.to_numeric(df_общий['Площадь'], errors='coerce')
has_missing = df_общий['Площадь'].isna().any()
print(f"Есть пропуски в столбце 'Площадь': {has_missing}")
missing_count = df_общий['Площадь'].isna().sum()
print(f"Количество пропусков в столбце 'Площадь': {missing_count}")

Есть пропуски в столбце 'Площадь': True
Количество пропусков в столбце 'Площадь': 71


In [95]:
# Удаляем строки с NaN
df_общий = df_общий.dropna(subset=['Площадь'])

In [96]:
# Рассчитываем только там, где оба значения не пустые
df_общий['Цена за метр'] = df_общий['Оценка цены'] / df_общий['Площадь']

In [100]:
# 2. Заполняем средним по полной группе (ЖК + Год + Месяц + Комнатность)
df_общий['Цена за метр'] = df_общий.groupby(['ЖК рус', 'Год', 'Месяц', 'Тип Комнатности'])['Цена за метр'].transform(
    lambda x: x.fillna(x.mean())
)

# 3. Если остались пропуски, заполняем средним по группе без комнатности
df_общий['Цена за метр'] = df_общий.groupby(['ЖК рус', 'Год', 'Месяц'])['Цена за метр'].transform(
    lambda x: x.fillna(x.mean())
)

# 4. Если всё еще есть пропуски, заполняем средним по ЖК
df_общий['Цена за метр'] = df_общий.groupby(['ЖК рус'])['Цена за метр'].transform(
    lambda x: x.fillna(x.mean())
)

print("Пропуски успешно заполнены!")
print(f"Осталось пропусков: {df_общий['Цена за метр'].isna().sum()}")

Пропуски успешно заполнены!
Осталось пропусков: 889


In [101]:
df_общий['Оценка цены'] = df_общий['Оценка цены'].fillna(df_общий['Цена за метр'] * df_общий['Площадь']).round(0)

In [103]:
df_общий = df_общий.dropna(subset=['Оценка цены']).reset_index(drop=True)

In [106]:
df_общий.drop(['Год', 'Месяц', 'Оценка по ЕИСЖС', 'Цена со скидкой',
         'Покупатель ЮЛ', 'Купил лотов в ЖК', 'Уступка'], axis=1, inplace=True)

In [109]:
df_общий['Цена за метр'] = df_общий['Цена за метр'].round(1)

In [139]:
df_общий

,ID ЖК,ЖК рус,Район Город,Округ Направление,Регион,АТД,Застройщик ЖК,Площадь,Кол-во комнат,Тип помещения,...,Дата регистрации,Залогодержатель,Тип обременения,Оценка цены,Ипотека,ID дом.рф,Цена за метр,Название проекта,Девелопер,Тип Комнатности пыпин
0,8539,А22,Даниловский,ЮАО,Москва,ЮАО,Консоль,37.1,1,квартира,...,2025-12-24,СБЕРБАНК,ипотека,19378629.0,1.0,64921,522335.0,А22,Консоль,1
1,8539,А22,Даниловский,ЮАО,Москва,ЮАО,Консоль,40.1,1,квартира,...,2025-11-05,NaN,NaN,21578612.0,0.0,64921,538120.0,А22,Консоль,1
2,8539,А22,Даниловский,ЮАО,Москва,ЮАО,Консоль,63.8,2,квартира,...,2025-09-12,NaN,NaN,36107610.0,0.0,64921,565950.0,А22,Консоль,2
3,8539,А22,Даниловский,ЮАО,Москва,ЮАО,Консоль,90.3,3,квартира,...,2025-09-11,NaN,NaN,47037722.0,0.0,64921,520905.0,А22,Консоль,3
4,8539,А22,Даниловский,ЮАО,Москва,ЮАО,Консоль,49.8,1,квартира,...,2025-09-11,NaN,NaN,27420129.0,0.0,64921,550605.0,А22,Консоль,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1234370,6659,Юнино,Щербинка,НАО,Новая Москва,НАО,ПИК,55.7,2,квартира,...,2023-02-13,СБЕРБАНК,ипотека,10499800.0,1.0,49819,188506.3,Юнино,ПИК,2
1234371,6659,Юнино,Щербинка,НАО,Новая Москва,НАО,ПИК,51.0,2,квартира,...,2023-02-09,NaN,NaN,9700880.0,0.0,49819,190213.3,Юнино,ПИК,2
1234372,6659,Юнино,Щербинка,НАО,Новая Москва,НАО,ПИК,36.0,1,квартира,...,2023-02-06,СБЕРБАНК,ипотека,7375341.0,1.0,49819,204870.6,Юнино,ПИК,1
1234373,6659,Юнино,Щербинка,НАО,Новая Москва,НАО,ПИК,32.0,1,квартира,...,2023-02-06,СБЕРБАНК,ипотека,6938631.0,1.0,49819,216832.2,Юнино,ПИК,1


In [128]:
import json

In [129]:
with open(r'C:\PycharmProjects\ndv_parcing\!haracteristik_dictionary\projects.json', 'r', encoding='utf-8') as f: json_data = json.load(f)

In [130]:
# 1. Создаем словарь для маппинга: ID ЖК -> (Название проекта, Девелопер)
mapping = {}
for project_name, project_info in json_data.items():
    project_id = project_info.get('id')
    developer = project_info.get('Девелопер')

    # Пропускаем, если id = 'nan' или None
    if project_id and project_id != 'nan' and pd.notna(project_id):
        mapping[project_id] = {
            'Название проекта': project_name,
            'Девелопер': developer
        }

{'6921': {'Название проекта': '1-й Донской', 'Девелопер': 'ФСК'}, '7268': {'Название проекта': '1-й Измайловский', 'Девелопер': 'ФСК'}, '5642': {'Название проекта': '1-й Ленинградский', 'Девелопер': 'ФСК'}, '5043': {'Название проекта': '1-й Лермонтовский', 'Девелопер': 'ФСК'}, '7622': {'Название проекта': '1-й Саларьевский', 'Девелопер': 'ФСК'}, '7409': {'Название проекта': '1-й Химкинский', 'Девелопер': 'ФСК'}, '6922': {'Название проекта': '1-й Шереметьевский', 'Девелопер': 'ФСК'}, '6923': {'Название проекта': '1-й Южный', 'Девелопер': 'ФСК'}, '7269': {'Название проекта': '1-й Ясеневский', 'Девелопер': 'ФСК'}, '7464': {'Название проекта': '2-й Иртышский', 'Девелопер': 'ПИК'}, '1281': {'Название проекта': '31 Квартал', 'Девелопер': 'ПРОФИ-Инвест'}, '7243': {'Название проекта': '7 небо', 'Девелопер': 'ПРОФИ-Инвест'}, '4945': {'Название проекта': '8 Кленов', 'Девелопер': 'Сити 21 век'}, '5095': {'Название проекта': 'AFI Park Воронцовский', 'Девелопер': 'AFI'}, '5538': {'Название проекта'

In [131]:
# 2. Создаем столбцы и заполняем
df_общий['Название проекта'] = df_общий['ID ЖК'].map(lambda x: mapping.get(x, {}).get('Название проекта', ''))
df_общий['Девелопер'] = df_общий['ID ЖК'].map(lambda x: mapping.get(x, {}).get('Девелопер', ''))

In [136]:
df_общий['Тип Комнатности пыпин'] = df_общий['Тип Комнатности']

In [137]:
df_общий = df_общий.rename(columns={'Тип Комнатности': 'Кол-во комнат'})

In [138]:
df_общий.to_csv(r"D:\Ricci\archive\Pipin-04-2026-2.csv", index=False, encoding='utf-8-sig')

In [4]:
df_new = pd.read_csv(r"D:\Ricci\archive\Pipin-04-2026-kvartirografia.csv")

In [5]:
df_classes = pd.read_excel(r"D:\Ricci\Риччи классы.xlsx")

In [6]:
df_classes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 667 entries, 0 to 666
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ЖК рус       667 non-null    object
 1   Класс Ricci  667 non-null    object
dtypes: object(2)
memory usage: 10.6+ KB


In [7]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1234375 entries, 0 to 1234374
Data columns (total 21 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   ID ЖК                  1234375 non-null  int64  
 1   ЖК рус                 1234375 non-null  object 
 2   Район Город            1234375 non-null  object 
 3   Округ Направление      1234375 non-null  object 
 4   Регион                 1234375 non-null  object 
 5   АТД                    1234375 non-null  object 
 6   Застройщик ЖК          1234375 non-null  object 
 7   Площадь                1234375 non-null  float64
 8   Кол-во комнат          1229305 non-null  object 
 9   Тип помещения          1234375 non-null  object 
 10  Корпус                 1223919 non-null  object 
 11  Дата регистрации       1234375 non-null  object 
 12  Залогодержатель        785430 non-null   object 
 13  Тип обременения        786238 non-null   object 
 14  Оценка цены       

In [9]:
result = pd.merge(df_new, df_classes, on='ЖК рус', how='left')
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1234375 entries, 0 to 1234374
Data columns (total 22 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   ID ЖК                  1234375 non-null  int64  
 1   ЖК рус                 1234375 non-null  object 
 2   Район Город            1234375 non-null  object 
 3   Округ Направление      1234375 non-null  object 
 4   Регион                 1234375 non-null  object 
 5   АТД                    1234375 non-null  object 
 6   Застройщик ЖК          1234375 non-null  object 
 7   Площадь                1234375 non-null  float64
 8   Кол-во комнат          1229305 non-null  object 
 9   Тип помещения          1234375 non-null  object 
 10  Корпус                 1223919 non-null  object 
 11  Дата регистрации       1234375 non-null  object 
 12  Залогодержатель        785430 non-null   object 
 13  Тип обременения        786238 non-null   object 
 14  Оценка цены       

In [14]:
# 2. Извлекаем год в новый столбец
result['Дата регистрации'] = pd.to_datetime(result['Дата регистрации'])
result['Год'] = result['Дата регистрации'].dt.year

missing_data = result[result['Класс Ricci'].isna()]

# 3. Считаем пропуски в столбце 'Класс Ricci' по годам
missing_by_year =(
    missing_data.groupby('Год')['ЖК рус']
    .agg(lambda x: list(x.unique()))  # или .agg(list) если нужны повторы
    .reset_index()
    .rename(columns={'ЖК рус': 'Список_ЖК_с_пропусками'})
)

print(missing_by_year)

     Год  \
0   2016   
1   2017   
2   2018   
3   2019   
4   2020   
5   2021   
6   2022   
7   2023   
8   2024   
9   2025   
10  2026   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       